# Семинар 5. Скорость сходимости градиентного спуска

Программная часть семинара. Первые пять задач — **основные**, задачи 6–10 — **дополнительные**.

Цель ноутбука — исследовать те же идеи, что и в теоретической части, на задачах, где ручные вычисления уже неудобны: сравнивать скорость сходимости, влияние шага, начального направления, регуляризации и критерия остановки.

Готовые оптимизаторы использовать не нужно: достаточно `numpy` и `matplotlib`.


## Подготовка

Вспомогательные функции ниже можно использовать во всех задачах. Они не содержат решений самих заданий, а только убирают повторяющийся технический код.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)


def make_spd(eigenvalues, rng=rng):
    """Создать симметричную положительно определённую матрицу с заданным спектром."""
    eigenvalues = np.asarray(eigenvalues, dtype=float)
    Q, _ = np.linalg.qr(rng.normal(size=(len(eigenvalues), len(eigenvalues))))
    return Q @ np.diag(eigenvalues) @ Q.T


def quadratic_value(A, x):
    return 0.5 * x @ A @ x


def gd_quadratic(A, x0, alpha, n_steps):
    """Траектория GD для f(x)=1/2 x^T A x."""
    x = np.asarray(x0, dtype=float).copy()
    xs = [x.copy()]
    for _ in range(n_steps):
        x = x - alpha * (A @ x)
        xs.append(x.copy())
    return np.asarray(xs)


def relative_quadratic_gaps(A, xs):
    vals = np.array([quadratic_value(A, x) for x in xs])
    if vals[0] == 0:
        return vals
    return vals / vals[0]


# Основные задачи


## Задача 1. Распознать тип сходимости по экспериментальным данным

Предположим, что три эксперимента дали последовательности относительных ошибок

$$r_k=\frac{f(x_k)-f^\star}{f(x_0)-f^\star}.$$

В этой задаче вам дана более длинная и слегка зашумлённая версия таких журналов.

**1.** Постройте для всех трёх последовательностей графики в обычном масштабе.

**2.** Постройте полулогарифмические графики: по вертикали используйте `log(r_k)`.

**3.** Постройте логарифмические графики: `log(r_k)` против `log(k)`.

**4.** Определите, какие последовательности больше похожи на геометрические, а какая — на степенную.

**5.** Для последовательностей, которые вы считаете геометрическими, оцените множитель `q` по наклону линейной регрессии для `log(r_k)`.

**6.** По найденному `q` спрогнозируйте номер итерации, на котором относительная ошибка впервые станет меньше `1e-4`.


In [ ]:
k = np.arange(1, 101)

noise_A = np.exp(rng.normal(0, 0.015, size=len(k)))
noise_B = np.exp(rng.normal(0, 0.015, size=len(k)))
noise_C = np.exp(rng.normal(0, 0.015, size=len(k)))

traces = {
    "A": (0.82 ** k) * noise_A,
    "B": (1.0 / (k + 1)) * noise_B,
    "C": (0.95 ** k) * noise_C,
}

# TODO: построить три типа графиков.
# TODO: оценить q для геометрических последовательностей.
# Подсказка: np.polyfit(k, np.log(r), 1) возвращает наклон прямой.


## Задача 2. Найти хороший постоянный шаг

Создадим квадратичную функцию в размерности `d=50`. Её собственные значения лежат между 1 и 9, но собственные направления случайно повернуты.

Для постоянного шага `alpha` теоретический множитель в худшем собственном направлении равен

$$\rho(\alpha)=\max_i|1-\alpha\lambda_i|.$$

**1.** Для сетки значений `alpha` вычислите `rho(alpha)`.

**2.** Для каждого `alpha` запустите GD из одной и той же начальной точки и найдите число итераций до достижения относительной ошибки `1e-8`.

**3.** Постройте два графика: `rho(alpha)` и фактическое число итераций от `alpha`.

**4.** Сравните положения минимумов двух графиков.

**5.** Отметьте на графиках шаг `1/L`. Является ли он лучшим постоянным шагом в этом эксперименте?

**6.** Объясните, что происходит при приближении `alpha` к границе устойчивости `2/L`.


In [ ]:
d = 50
eigenvalues = np.linspace(1.0, 9.0, d)
A2 = make_spd(eigenvalues)
x0_2 = rng.normal(size=d)
mu2, L2 = eigenvalues.min(), eigenvalues.max()

alphas = np.linspace(0.01, 0.99 * (2.0 / L2), 120)
tol = 1e-8
max_steps = 5000

# TODO: вычислить rho(alpha) для каждого alpha.
# TODO: для каждого alpha измерить число итераций до relative gap <= tol.
# TODO: построить два графика и отметить alpha = 1/L2.


## Задача 3. Начальное направление и фактическая скорость

Рассмотрим диагональную квадратичную задачу размерности 30 со спектром от 1 до 100 и шагом `alpha=1/L`.

Теоретическая оценка через `kappa` является худшей гарантией. В этой задаче нужно выяснить, насколько сильно реальное число итераций зависит от начального направления.

**1.** Сгенерируйте не менее 1500 случайных единичных начальных векторов.

**2.** Для каждого старта найдите число итераций до относительной ошибки `1e-8`.

**3.** Для каждого старта вычислите абсолютную проекцию на собственный вектор, соответствующий `lambda_min`.

**4.** Постройте диаграмму рассеяния: по горизонтали — величина этой проекции, по вертикали — число итераций.

**5.** Найдите самый быстрый и самый медленный старты. Сравните их координаты в собственном базисе.

**6.** Объясните, почему большое `kappa` не означает, что любой запуск обязательно будет медленным.


In [ ]:
d = 30
eigs3 = np.geomspace(1.0, 100.0, d)
A3 = np.diag(eigs3)
alpha3 = 1.0 / eigs3.max()

n_starts = 1500
tol3 = 1e-8
max_steps3 = 5000

starts = rng.normal(size=(n_starts, d))
starts /= np.linalg.norm(starts, axis=1, keepdims=True)

# Для диагональной A3 собственный вектор lambda_min — первый координатный вектор.
slow_projection = np.abs(starts[:, 0])

# TODO: измерить число итераций для каждого старта.
# TODO: построить scatter(slow_projection, iterations).
# TODO: исследовать самый быстрый и самый медленный старты.


## Задача 4. Ridge: обусловленность против числа итераций

Создадим плохо обусловленную задачу наименьших квадратов

$$F_\lambda(w)=\frac12\|Xw-y\|_2^2+\frac{\lambda}{2}\|w\|_2^2.$$

Её гессиан равен

$$H_\lambda=X^\top X+\lambda I.$$

**1.** Для сетки `lambda` вычислите `kappa(H_lambda)`.

**2.** Для каждого `lambda` найдите точное решение через `np.linalg.solve`.

**3.** Запустите GD с шагом `1/L_lambda` из нуля и измерьте число итераций до относительной ошибки по функции `1e-8`.

**4.** Постройте графики `lambda -> kappa` и `lambda -> iterations`.

**5.** Проверьте, действительно ли уменьшение `kappa` сопровождается ускорением GD.

**6.** Почему из этого эксперимента нельзя сделать вывод «чем больше lambda, тем лучше модель»?


In [ ]:
n, d = 240, 20
Q_left, _ = np.linalg.qr(rng.normal(size=(n, d)))
Q_right, _ = np.linalg.qr(rng.normal(size=(d, d)))
singular_values = np.geomspace(0.08, 3.0, d)
X4 = Q_left @ np.diag(singular_values) @ Q_right.T
y4 = rng.normal(size=n)

lambdas = np.logspace(-4, 2, 35)
tol4 = 1e-8
max_steps4 = 20000

# TODO: для каждого lambda построить H = X.T @ X + lambda * I.
# TODO: вычислить kappa(H), точное w_star и число итераций GD.
# Градиент: X.T @ (X @ w - y) + lambda * w


## Задача 5. Один критерий остановки — разное качество решения

Сравним три квадратичные задачи с разной минимальной кривизной. Для каждой используйте критерий остановки

$$\|\nabla f(x_k)\|_2\le 10^{-3}.$$

**1.** Для каждой матрицы запустите GD с шагом `1/L` из одной и той же начальной точки.

**2.** В момент остановки сохраните три величины: норму градиента, ошибку по аргументу `||x_k-x_star||` и ошибку по функции `f(x_k)-f_star`.

**3.** Составьте таблицу результатов.

**4.** Сравните, насколько разным оказывается качество решения при одном и том же пороге на градиент.

**5.** Свяжите наблюдение с минимальным собственным значением `mu` каждой задачи.


In [ ]:
matrices5 = {
    "очень плоская": np.diag([1e-3, 1.0]),
    "умеренная": np.diag([1.0, 2.0]),
    "крутая": np.diag([100.0, 200.0]),
}

x0_5 = np.array([1.0, 1.0])
grad_tol = 1e-3
max_steps5 = 50000

# TODO: для каждой матрицы запустить GD с alpha = 1/L.
# TODO: остановиться при ||A @ x|| <= grad_tol.
# TODO: сохранить число шагов, ||grad||, ||x|| и f(x).


# Дополнительные задачи


## Задача 6. Масштабирование функции не меняет траекторию при согласованном шаге

Возьмите одну SPD-квадратичную функцию и рассмотрите масштабированные функции `g_c(x)=c f(x)` для нескольких `c`.

**1.** Для каждого `c` используйте шаг `1/(cL)`.

**2.** Запустите GD из одной начальной точки.

**3.** Наложите траектории координат друг на друга.

**4.** Численно оцените максимальное расстояние между соответствующими итерациями разных запусков.

**5.** Проверьте, что число обусловленности не зависит от `c`.


In [ ]:
eigs6 = np.array([1.0, 3.0, 12.0])
A6 = make_spd(eigs6)
x0_6 = np.array([1.0, -2.0, 0.5])
scales6 = [1.0, 10.0, 1000.0]

# TODO: запустить GD для c*A6 с шагом 1/(c*L).
# TODO: сравнить траектории и числа обусловленности.


## Задача 7. Инвариантность к ортогональному повороту

Сравните задачи с матрицами

$$A=\operatorname{diag}(1,20),\qquad B=QAQ^\top,$$

где `Q` — матрица поворота.

**1.** Проверьте совпадение спектров и чисел обусловленности.

**2.** Возьмите `y0 = Q @ x0` и запустите GD на обеих задачах с шагом `1/L`.

**3.** Проверьте численно, что `y_k` совпадает с `Q @ x_k`.

**4.** Сравните последовательности ошибок по функции.

**5.** Нарисуйте линии уровня обеих функций и траектории GD.


In [ ]:
theta = 0.7
Q7 = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])
A7 = np.diag([1.0, 20.0])
B7 = Q7 @ A7 @ Q7.T
x0_7 = np.array([2.0, 1.0])
y0_7 = Q7 @ x0_7

# TODO: сравнить спектры, kappa, траектории и gaps.
# TODO: визуализировать линии уровня и траектории.


## Задача 8. Брутфорсом найти самый медленный старт

Пусть

$$A=\operatorname{diag}(1,4,10),\qquad \alpha=0.1.$$

**1.** Сгенерируйте не менее 5000 случайных единичных стартов.

**2.** Для каждого вычислите `||e_k||` после `k=20` шагов.

**3.** Найдите старт с максимальным `||e_k||`.

**4.** Сравните его направление с собственным вектором для `lambda_min=1`.

**5.** Повторите эксперимент для `k=1`. Совпадает ли худший старт?


In [ ]:
A8 = np.diag([1.0, 4.0, 10.0])
alpha8 = 0.1
n_samples8 = 5000

starts8 = rng.normal(size=(n_samples8, 3))
starts8 /= np.linalg.norm(starts8, axis=1, keepdims=True)

# TODO: оценить худший старт после k=20 и k=1.


## Задача 9. «Эффективное» число обусловленности по наблюдаемой скорости

Создайте SPD-квадратичную задачу с истинным числом обусловленности около 50. Для разных начальных точек оцените поздний множитель

$$q_{\mathrm{emp}}\approx\frac{f(x_{k+1})-f^\star}{f(x_k)-f^\star}.$$

После этого определите

$$\kappa_{\mathrm{emp}}=\frac{1}{1-q_{\mathrm{emp}}}.$$

**1.** Сравните `kappa_emp` для разных стартов с истинным `kappa`.

**2.** Постройте распределение `kappa_emp`.

**3.** Найдите старт, для которого оценка ближе всего к истинной `kappa`.

**4.** Исследуйте его проекцию на медленное собственное направление.

**5.** Объясните, почему по одному запуску нельзя надёжно восстановить истинное число обусловленности.


In [ ]:
d = 20
eigs9 = np.geomspace(1.0, 50.0, d)
A9 = np.diag(eigs9)
alpha9 = 1.0 / eigs9.max()
true_kappa9 = eigs9.max() / eigs9.min()

n_starts9 = 300
starts9 = rng.normal(size=(n_starts9, d))
starts9 /= np.linalg.norm(starts9, axis=1, keepdims=True)

# TODO: для каждого старта оценить q_emp на поздних итерациях.
# Совет: избегайте итераций, где gap уже упал до машинного нуля.
# TODO: вычислить kappa_emp и сравнить с true_kappa9.


## Задача 10. Проверить спорные утверждения экспериментом

Для каждого утверждения постройте небольшой численный эксперимент, который его подтверждает или опровергает.

**а)** «Если `L` меньше, GD всегда сходится быстрее».

**б)** «Если `kappa=100`, ошибка на каждой итерации обязательно умножается на `0.99`».

**в)** «Если `||grad f(x)||` мало, то `x` обязательно близко к `x_star`».

**г)** «Ортогональный поворот линий уровня может ухудшить число обусловленности».

**д)** «Добавление Ridge-регуляризации может улучшить обусловленность».

Для каждого пункта сохраните конкретный численный пример и коротко сформулируйте вывод под соответствующей ячейкой.


In [ ]:
# а) TODO: придумайте две задачи, у которых сравнение L не совпадает со сравнением скорости.


In [ ]:
# б) TODO: задача с kappa=100 и стартом, не содержащим медленного собственного направления.


In [ ]:
# в) TODO: очень плоская квадратичная функция.


In [ ]:
# г) TODO: сравнить A и Q @ A @ Q.T.


In [ ]:
# д) TODO: сравнить kappa(H) и kappa(H + lambda * I).
